# Traditional RAG (Retrieval-Augmented Generation)

Traditional RAG retrieves relevant information from a **vector store** and provides it to an LLM as context to generate an accurate, relevant answer.

---

## 1. RAG Workflow

1. **User Query:** User asks a question (e.g., "What is the leave policy of my company?").
2. **Retrieval:** The system searches the vector store for relevant information.
3. **Prompting:** Retrieved information is added to the prompt as context, along with instructions and the user's question.
4. **Generation:** The LLM uses the provided context to generate an answer.

![Traditional RAG Workflow](Images/rag_workflow.jpg)

---

## 2. Main Components

- **Retrieval:** Finds relevant information from the vector store based on the user's query.
- **Context:** Relevant retrieved documents or text passed to the LLM.
- **Prompting:** Combines the context, user query, and instructions into a prompt.
- **Generation:** The LLM generates the final response using the provided context.

**Key Point:** RAG connects external knowledge to an LLM, helping it answer questions using relevant information instead of relying only on its pretrained knowledge.

![Pipelines](Images/pipelines.png)


<br><br>

>### Data Ingestion Pipeling

![Pipelines](Images/Data_Ingestion.png)


<br><br>


In [12]:
from langchain_core.documents import Document

doc = Document(
    page_content="This is a test document for Rag.",
    metadata={
        "source": "test.txt",
        "page": 1,
        "content": "This is a test document for Rag.",
        "date": "2024-06-01",
        })

doc

Document(metadata={'source': 'test.txt', 'page': 1, 'content': 'This is a test document for Rag.', 'date': '2024-06-01'}, page_content='This is a test document for Rag.')

In [13]:
from pathlib import Path

# Use the Rag folder even if the notebook kernel starts from the repository root.
rag_dir = Path.cwd()
if rag_dir.name.lower() != "rag":
    rag_dir = rag_dir / "Rag"

test_files_dir = rag_dir / "data" / "test_files"
test_files_dir.mkdir(parents=True, exist_ok=True)

sample_texts = {
    "rag_overview.txt": """Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation combines information retrieval with a large language model. Instead of relying only on knowledge learned during training, a RAG system searches a knowledge base for relevant text and supplies that text to the model as context. This can reduce unsupported answers and allows the application to use documents that may be newer or private.

A traditional RAG workflow loads source documents, converts them into LangChain Document objects, divides them into chunks, creates vector embeddings, stores those embeddings, and uses relevant chunks as context for a language model.

Good chunking preserves enough context to answer a question without making chunks unnecessarily large. Source metadata such as file name and page number helps users verify retrieved answers.""",
    "document_loaders.txt": """LangChain Document Loaders

Document loaders read data from external sources and convert it into LangChain Document objects. A Document has page_content for the text and metadata for details such as source file and page number.

TextLoader reads plain text, PyPDFLoader reads PDF pages, CSVLoader reads rows from CSV files, WebBaseLoader retrieves website content, and Docx2txtLoader extracts text from Word documents. DirectoryLoader loads several files from a folder.

Loaders perform loading only. Splitting, embedding, and vector storage are separate pipeline stages.""",
    "text_splitting.txt": """Text Splitting and Chunking

Text splitting divides source documents into smaller pieces suitable for retrieval and embedding. Chunk size controls the maximum text in a piece, while chunk overlap controls how much text is repeated in adjacent pieces.

RecursiveCharacterTextSplitter tries paragraph, sentence, and word boundaries before smaller character boundaries. Small chunks can retrieve precise facts but may omit surrounding context. Large chunks contain more context but may include unrelated text.

Applications should preserve metadata such as file name, page number, section name, and chunk position so retrieved answers can be traced to their source."""
}

for file_name, content in sample_texts.items():
    (test_files_dir / file_name).write_text(content, encoding="utf-8")

print(f"? Sample text files created in: {test_files_dir}")


? Sample text files created in: e:\Learning\Generative Ai\Rag\data\test_files


In [ ]:
# text loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    str(test_files_dir / "rag_overview.txt"),
    encoding="utf-8"
)

document = loader.load()
print(document)


In [17]:
# directory loader

from langchain_community.document_loaders import DirectoryLoader

## loading all text files in the test_files_dir

dir_loader = DirectoryLoader(
    str(test_files_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

document = dir_loader.load()
document

[Document(metadata={'source': 'e:\\Learning\\Generative Ai\\Rag\\data\\test_files\\document_loaders.txt'}, page_content='LangChain Document Loaders\n\nDocument loaders read data from external sources and convert it into LangChain Document objects. A Document has page_content for the text and metadata for details such as source file and page number.\n\nTextLoader reads plain text, PyPDFLoader reads PDF pages, CSVLoader reads rows from CSV files, WebBaseLoader retrieves website content, and Docx2txtLoader extracts text from Word documents. DirectoryLoader loads several files from a folder.\n\nLoaders perform loading only. Splitting, embedding, and vector storage are separate pipeline stages.'),
 Document(metadata={'source': 'e:\\Learning\\Generative Ai\\Rag\\data\\test_files\\rag_overview.txt'}, page_content='Retrieval-Augmented Generation (RAG)\n\nRetrieval-Augmented Generation combines information retrieval with a large language model. Instead of relying only on knowledge learned durin

In [ ]:
# PDF loader

from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

rag_dir = Path.cwd()
if rag_dir.name.lower() != "rag":
    rag_dir = rag_dir / "Rag"

pdf_dir = rag_dir / "data" / "pdf"
print(f"PDF folder: {pdf_dir}")

pdf_loader = DirectoryLoader(
    str(pdf_dir),
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
)

all_pdf_documents = pdf_loader.load()
print(f"Loaded PDF pages/documents: {len(all_pdf_documents)}")
all_pdf_documents[:3]


# PDF Text Splitter

The loaded PDF pages are split into smaller chunks so they are suitable for embeddings and retrieval.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(all_pdf_documents)
print(f"Created chunks: {len(chunks)}")
chunks[:3]


# Embeddings

An embedding model converts each text chunk into a numeric vector. Similar meaning generally produces vectors close together. Embeddings are used in the same way for documents and questions, but a document chunk and a question are passed through the same model so their vectors occupy the same vector space.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

embedding_vector = embeddings.embed_query("How are RAG documents retrieved?")
print(f"Embedding dimensions: {len(embedding_vector)}")
print(f"First values: {embedding_vector[:5]}")


# Vector database: Chroma

Chroma stores chunk text, its embedding, and metadata such as source file and page number. When a query arrives, Chroma embeds the query and performs similarity search. The returned chunks become the retrieval context for an LLM.

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma

rag_dir = Path.cwd()
if rag_dir.name.lower() != "rag":
    rag_dir = rag_dir / "Rag"

vector_store_dir = rag_dir / "data" / "chroma_db"
vector_store_dir.mkdir(parents=True, exist_ok=True)

vector_store = Chroma(
    collection_name="rag_study_documents",
    embedding_function=embeddings,
    persist_directory=str(vector_store_dir),
)

print(f"Existing vectors: {vector_store._collection.count()}")


In [ ]:
# Add chunks only once by clearing the collection during this learning exercise.
vector_store.reset_collection()
vector_store.add_documents(chunks)
print(f"Stored vectors: {vector_store._collection.count()}")


# Retrieval

Similarity retrieval returns the chunks nearest to the query vector. `k` controls how many chunks are retrieved. Metadata and source citations remain attached to each result.

In [ ]:
query = "How does text chunk overlap preserve context?"
retrieved_chunks = vector_store.similarity_search(query, k=3)

print(f"Query: {query}")
for index, result in enumerate(retrieved_chunks, start=1):
    print(f"\nResult {index}:")
    print(result.metadata)
    print(result.page_content[:500])
